# Solución · Planificador de producción

Este notebook resuelve las tres consignas de "Para expandir" del programa base:

1. Precio de venta y margen esperado.
2. Tres objetos `Produccion` con distintas cantidades de unidades, comparando su viabilidad.
3. Método que calcula la cantidad máxima de unidades según el tiempo disponible.

La clase se reescribe para que `horas_por_unidad` y `valor_hora` pasen a ser atributos del objeto (en lugar de parámetros repetidos en cada llamada), y para que cada cálculo tenga un único método responsable. La creación de las tres producciones se resuelve con una función fábrica, para no repetir la carga de materiales tres veces.

## 1. Clase `Produccion` extendida

Cambios respecto del programa base:

- `horas_por_unidad` y `valor_hora` ahora son atributos del objeto: cada producción puede tener su propio ritmo de trabajo y costo de mano de obra, y los métodos ya no necesitan recibirlos como parámetro en cada llamada.
- `precio_venta_unidad` es un atributo nuevo, con 0 por defecto. A partir de él, `calcular_ingreso_total()`, `calcular_margen()` y `calcular_margen_porcentual()` resuelven la consigna 1.
- `calcular_unidades_maximas_por_tiempo()` resuelve la consigna 3: divide las horas disponibles por las horas que insume cada unidad.
- `es_viable_presupuesto()` y `es_viable_tiempo()` separan cada condición, y `evaluar()` los combina; así se pueden reutilizar por separado (por ejemplo, en la comparación).
- `generar_informe()` arma el reporte completo como texto, igual que en la solución del programa 1, para no repetir los `print()` por cada objeto.

In [ ]:
class Produccion:
    def __init__(
        self,
        nombre: str,
        unidades: int,
        presupuesto: float,
        horas_disponibles: float,
        horas_por_unidad: float,
        valor_hora: float,
        precio_venta_unidad: float = 0.0,
    ) -> None:
        self.nombre: str = nombre
        self.unidades: int = unidades
        self.presupuesto: float = presupuesto
        self.horas_disponibles: float = horas_disponibles
        self.horas_por_unidad: float = horas_por_unidad
        self.valor_hora: float = valor_hora
        self.precio_venta_unidad: float = precio_venta_unidad
        self.materiales: list[tuple[str, float, float]] = []

    def agregar_material(
        self,
        nombre: str,
        cantidad_por_unidad: float,
        precio: float,
    ) -> None:
        self.materiales.append((nombre, cantidad_por_unidad, precio))

    def calcular_materiales(self) -> float:
        total: float = 0.0
        for _, cantidad, precio in self.materiales:
            total += cantidad * precio * self.unidades
        return total

    def calcular_mano_obra(self) -> float:
        return self.horas_por_unidad * self.valor_hora * self.unidades

    def calcular_total(self) -> float:
        return self.calcular_materiales() + self.calcular_mano_obra()

    def calcular_horas_necesarias(self) -> float:
        return self.horas_por_unidad * self.unidades

    def calcular_unidades_maximas_por_tiempo(self) -> int:
        # Cuántas unidades entran en las horas disponibles, sin pasarse.
        if self.horas_por_unidad <= 0:
            return 0
        return int(self.horas_disponibles // self.horas_por_unidad)

    def calcular_ingreso_total(self) -> float:
        return self.precio_venta_unidad * self.unidades

    def calcular_margen(self) -> float:
        return self.calcular_ingreso_total() - self.calcular_total()

    def calcular_margen_porcentual(self) -> float:
        ingreso: float = self.calcular_ingreso_total()
        if ingreso <= 0:
            return 0.0
        return (self.calcular_margen() / ingreso) * 100

    def es_viable_presupuesto(self) -> bool:
        return self.calcular_total() <= self.presupuesto

    def es_viable_tiempo(self) -> bool:
        return self.calcular_horas_necesarias() <= self.horas_disponibles

    def evaluar(self) -> str:
        presupuesto_ok: bool = self.es_viable_presupuesto()
        tiempo_ok: bool = self.es_viable_tiempo()

        if presupuesto_ok and tiempo_ok:
            return "Producción viable por presupuesto y tiempo"
        if not presupuesto_ok and not tiempo_ok:
            return "Producción no viable: revisar presupuesto y tiempo"
        if not presupuesto_ok:
            return "Producción no viable: revisar presupuesto"
        return "Producción no viable: revisar tiempo disponible"

    def generar_informe(self) -> str:
        lineas: list[str] = []
        lineas.append(f"PRODUCCIÓN: {self.nombre}")
        lineas.append(f"Unidades: {self.unidades}")
        lineas.append("-" * 52)

        for nombre, cantidad, precio in self.materiales:
            parcial: float = cantidad * precio * self.unidades
            lineas.append(f"{nombre:<18} ${parcial:>14,.2f}")

        lineas.append("-" * 52)
        lineas.append(f"Materiales:            ${self.calcular_materiales():>14,.2f}")
        lineas.append(f"Mano de obra:          ${self.calcular_mano_obra():>14,.2f}")
        lineas.append(f"Costo total:           ${self.calcular_total():>14,.2f}")
        lineas.append(f"Presupuesto:           ${self.presupuesto:>14,.2f}")
        lineas.append(f"Horas necesarias:      {self.calcular_horas_necesarias():>14.1f}")
        lineas.append(f"Horas disponibles:     {self.horas_disponibles:>14.1f}")
        lineas.append(
            f"Unid. máx. por tiempo: {self.calcular_unidades_maximas_por_tiempo():>14}"
        )
        lineas.append(f"Precio de venta (u.):  ${self.precio_venta_unidad:>14,.2f}")
        lineas.append(f"Ingreso total:         ${self.calcular_ingreso_total():>14,.2f}")
        lineas.append(
            f"Margen esperado:       ${self.calcular_margen():>14,.2f} "
            f"({self.calcular_margen_porcentual():.1f}%)"
        )
        lineas.append(f"Estado: {self.evaluar()}")

        return "\n".join(lineas)

## 2. Creación de tres producciones

`crear_produccion_lampara()` es una función fábrica: arma un objeto `Produccion` completo (con sus materiales ya cargados) a partir de una sola cantidad de unidades. Así, generar tres producciones para comparar es una lista por comprensión, y agregar una cuarta o una quinta no exige repetir la carga de materiales.

In [ ]:
def crear_produccion_lampara(unidades: int) -> Produccion:
    produccion = Produccion(
        nombre=f"Lámpara modular (x{unidades})",
        unidades=unidades,
        presupuesto=1_250_000.0,
        horas_disponibles=60.0,
        horas_por_unidad=2.2,
        valor_hora=7_500.0,
        precio_venta_unidad=65_000.0,
    )
    produccion.agregar_material("Madera", 0.8, 24_000.0)
    produccion.agregar_material("Cable", 1.5, 2_800.0)
    produccion.agregar_material("Portalámpara", 1.0, 5_600.0)
    return produccion


producciones: list[Produccion] = [
    crear_produccion_lampara(unidades) for unidades in (20, 27, 35)
]

print(f"Objetos creados: {[produccion.nombre for produccion in producciones]}")

## 3. Informe individual

Como en la solución del programa 1, mostrar el reporte de cualquier producción es siempre la misma línea de código gracias a `generar_informe()`.

In [ ]:
for produccion in producciones:
    print(produccion.generar_informe())
    print()

## 4. Comparación de viabilidad

`comparar_producciones()` acepta cualquier cantidad de objetos `Produccion` (no solo tres) y muestra, para cada uno, cuántas unidades entrarían en el tiempo disponible frente a las unidades planificadas.

In [ ]:
def comparar_producciones(*producciones: Produccion) -> None:
    if not producciones:
        print("No hay producciones para comparar.")
        return

    encabezado: str = (
        f"{'PRODUCCIÓN':<24}{'UNID.':>7}{'UNID. MÁX.':>12}"
        f"{'COSTO TOTAL':>16}{'MARGEN':>16}   {'ESTADO'}"
    )
    print(encabezado)
    print("-" * len(encabezado))

    for produccion in producciones:
        costo: float = produccion.calcular_total()
        margen: float = produccion.calcular_margen()
        unidades_max: int = produccion.calcular_unidades_maximas_por_tiempo()

        print(
            f"{produccion.nombre:<24}"
            f"{produccion.unidades:>7}"
            f"{unidades_max:>12}"
            f"${costo:>14,.2f}"
            f"${margen:>14,.2f}"
            f"   {produccion.evaluar()}"
        )

    mejor_margen: Produccion = max(producciones, key=lambda produccion: produccion.calcular_margen())
    print("-" * len(encabezado))
    print(f"Producción con mayor margen esperado: {mejor_margen.nombre}")


comparar_producciones(*producciones)

## Conclusiones

- El **margen esperado** surge de comparar `calcular_ingreso_total()` (precio de venta × unidades) contra `calcular_total()` (materiales + mano de obra). Con estos datos el margen resulta positivo y se mantiene en torno al 30% en las tres producciones, porque el precio de venta y el costo escalan igual por unidad; lo que distingue a las producciones no es el margen porcentual sino si el presupuesto y el tiempo alcanzan para sostenerlo a esa escala.
- La **cantidad máxima de unidades según el tiempo disponible** sale de dividir `horas_disponibles` por `horas_por_unidad` (`calcular_unidades_maximas_por_tiempo()`). Con los datos usados, el máximo es 27 unidades: por eso las producciones de 20 y 27 son viables tanto por tiempo como por presupuesto, mientras que la de 35 no es viable por ninguno de los dos criterios.
- La **comparación entre producciones** se resolvió igual que en la solución del programa 1: una función que recibe `*producciones` para poder comparar cualquier cantidad de objetos sin cambiar el código, más una función fábrica que evita repetir la carga de materiales en cada objeto nuevo.